# 融合算子：RMSNorm 优化

RMSNorm 是 Transformer 中高频出现的基础算子：每个 block 的 `attention_norm`、`ffn_norm`、`q_norm`、`k_norm` 以及模型末尾的 `norm` 都会调用一次。Qwen3-1.7B 共有 113 个 RMSNorm 实例（28 个 block × 4 + 1），是一个典型的"单算子开销小、但调用次数多"的热点。

本章以当前 Qwen3, seq_len=1024 的训练任务中典型的 Tensor shape $(2, 1024, 2048)$ 为例，先从概念层面分析融合的收益来源，再给出在 torchtitan-npu 中注册融合算子的完整流程，以便为下一章具体的 profiling 实测打下理论基础。

---

## 1. RMSNorm vs NPURMSNorm 收益分析

### RMSNorm 的数学公式

对最后一维长度为 $d$ 的向量 $x=(x_1,…,x_d)$，先计算均方值：

$$m=\frac{1}{d}\sum_{j=1}^{d}x_j^2$$

再计算逆均方根：

$$r=\frac{1}{\sqrt{m+\varepsilon}}$$

最后用可训练权重 $\gamma$ 对每个位置缩放：

$$y_i=\gamma_i x_i r=\gamma_i\frac{x_i}{\sqrt{\frac{1}{d}\sum_{j=1}^{d}x_j^2+\varepsilon}}$$

$\varepsilon$ 为自定义极小值（1e-5），用于保证分母稳定。

### 原生路径：公式展开为多个通用 op

原生 RMSNorm 调用 F.rms_norm 并 dispatch 到 aten::rms_norm，实际下发的是多个通用 device op（如平方、mean、rsqrt、乘法等），在 BF16 输入下还可能插入显式的类型转换 op。

多个通用 op 带来的主要开销有两类：一是每个 Kernel 都需要独立的 Host 下发与同步等待；二是中间结果（平方值、均值、逆平方根等）需反复写回 HBM 再被下一 op 读入，产生冗余的片外访存流量。

### 融合路径：一个融合 API 完成 RMSNorm

`torch_npu.npu_rms_norm` 接口把 `aclnnRmsNorm` 融合算子 暴露为一个融合 API 调用。
> API 文档：[`torch_npu.npu_rms_norm`](https://www.hiascend.com/document/detail/zh/Pytorch/730/apiref/torchnpuCustomsapi/docs/context/（beta）torch_npu-npu_rms_norm.md)

融合实现将 RMSNorm 所需的全部计算步骤合并为单次 Kernel Launch，以此消除多个通用 op 逐一下发的调度开销；同时利用片上 Unified Buffer（UB） 完成平方和、归约与缩放的完整链条，中间结果（平方值、均值等）全程驻留 UB，仅读写原始输入 x 和最终输出 y，从而减少 HBM 冗余访问。此外，aclnnRmsNorm 利用 RMSNorm 行间独立（各行 token 的归一化互不依赖）的特性，通过 Tiling 分块将张量按完整行切分搬入 UB，保证单行 Reduce 所需元素均在片上。

下面以典型 Tensor shape $(2, 1024, 2048)$ 为例，量化估算原生与融合路径在显存申请量与上的差异。

### 内存吞吐减少量分析

当前训练采用 BF16 输入，但原生 RMSNorm 为保证数值精度，将归一化计算提升至 FP32 进行。以形状为 `(2, 1024, 2048)` 的张量为例，其总元素数为 4,194,304：BF16 格式下占 8 MiB，转换为 FP32 后占 16 MiB；而按行归约后得到的 `(2, 1024, 1)` 小张量仅需 8 KiB。

依据 baseline trace 中的 8 段执行流水，可将原生路径的显存申请拆解为如下中间产物链：输入类型提升（BF16 → FP32）、逐元素平方、求均值（mean）、`rsqrt`、归一化后的逐元素乘回、与 `gamma` 权重做缩放乘，以及最终写回 BF16 输出。其中 `add_(eps)` 为原位（in-place）操作，不会额外申请新张量。累加上述各环节，原生路径的单次前向累计申请量约为 **72 MiB + 16 KiB**。

融合路径则更为精简：仅显式输出 BF16 的结果 `y`（8 MiB）与供反向传播使用的 FP32 中间量 `rstd`（8 KiB），累计申请量仅约 **8 MiB + 8 KiB**。因此，单次完整尺寸的前向传播中，融合算子可减少约 **64 MiB + 8 KiB** 的临时张量申请吞吐。


### Kernel Launch 开销减少收益分析

展开 baseline trace 中 `aten::rms_norm` 的嵌套，可以清晰地看到 8 个 ACLNN 调用：

1. `aclnnInplaceCopy`：BF16 输入转为 FP32；
2. `aclnnPowTensorScalar`：平方；
3. `aclnnMean`：沿 hidden dim 求均值；
4. `aclnnInplaceAdds`：加 epsilon；
5. `aclnnRsqrt`：求逆平方根；
6. `aclnnMul`：输入乘以 rstd；
7. `aclnnMul`：乘以 gamma；
8. `aclnnInplaceCopy`：FP32 输出转回 BF16。


- **头尾两个** `aclnnInplaceCopy`：负责 BF16 ↔ FP32 的输入输出类型转换；
- **中间 6 个**：依次完成平方（`aclnnPowTensorScalar`）、求均值（`aclnnMean`）、原地加 epsilon（`aclnnInplaceAdds`，不新增张量）、求逆平方根（`aclnnRsqrt`）、输入乘以 rstd（`aclnnMul`），以及乘以 gamma（`aclnnMul`）。

融合路径则将上述整条流水压缩为单个 `aclnnRmsNorm`，外层调用从 8 次 launch 减为 1 次。就当前 CANN/torch_npu 版本、shape 和 dtype 的实测 trace 而言，前向可稳定减少 7 次 ACLNN 下发。

下面，我们构造实验脚本验证。实验 1 基于上述流水，逐段累加临时张量; 实验 2 通过标量 add 构造 8 次串行 launch 与 1 次 launch 的实验。由于标量 add 本身几乎不占 device time，上述实验测得的时间差值主要反映 8 次 launch 与 1 次 launch 之间的下发路径开销差异，可作为 aclnnRmsNorm 融合收益中 launch 减少带来的参考上界。

In [ ]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/torchtitan-npu
%env BASH_ENV=/home/developer/Ascend/cann/set_env.sh

In [ ]:
# 1. 基础形状与张量大小定义
# B: batch size, S: sequence length, H: hidden dimension (对应 Qwen3-1.7B 的 hidden_size)
# 该 shape 用于模拟 RMSNorm 前向过程中的中间张量申请量
B, S, H = 2, 1024, 2048
elements = B * S * H        # 全张量元素总数：4,194,304
rows = B * S                # 归约（reduce）维度大小：沿 H 维求均值，输出张量的行数

# 存储单位转换常量
mib = 1024**2               # MiB 字节数
kib = 1024                  # KiB 字节数

# BF16 与 FP32 全尺寸张量及行向量的字节数
bf16_full = elements * 2    # BF16 每个元素占 2 字节 → 8 MiB
fp32_full = elements * 4    # FP32 每个元素占 4 字节 → 16 MiB
fp32_row = rows * 4         # 按 H 维归约后的 FP32 向量 → 8 KiB


# 2. 原生路径 vs 融合路径的理论累积申请量（累加和，非瞬时峰值）
# 注意：这里统计的是“累积申请量”，
# 即每次前向传播中所有中间张量申请字节数的总和，不代表瞬时显存占用量。
#
# 原生路径拆解（依据 Baseline Trace 的 8 个 ACLNN 调用）：
#   - input_fp32      : 输入从 BF16 提升至 FP32
#   - squared_fp32    : 逐元素平方
#   - mean_fp32       : 沿 H 维求均值
#   - rstd_fp32       : 求逆平方根（1 / sqrt(mean + eps)）
#   - normalized_fp32 : 原始输入乘以 rstd
#   - weighted_fp32   : 乘以 gamma 权重
#   - output_bf16     : 最终结果转回 BF16 输出
# 其中 aclnnInplaceAdds（加 eps）是原地操作，不产生新张量，故不纳入。
native_allocations = {
    'input_fp32': fp32_full,
    'squared_fp32': fp32_full,
    'mean_fp32': fp32_row,
    'rstd_fp32': fp32_row,
    'normalized_fp32': fp32_full,
    'weighted_fp32': fp32_full,
    'output_bf16': bf16_full,
}

# 融合路径（单个 aclnRmsNorm）仅显式输出两个张量：
#   - output_bf16 : 归一化后的 BF16 结果
#   - rstd_fp32   : 供反向传播使用的 FP32 逆标准差
fused_allocations = {'output_bf16': bf16_full, 'rstd_fp32': fp32_row}

# 计算各自累计申请量及理论节省值
native_bytes = sum(native_allocations.values())
fused_bytes = sum(fused_allocations.values())
saved_bytes = native_bytes - fused_bytes

# 打印结果
print(f'BF16 full tensor : {bf16_full / mib:.0f} MiB')
print(f'FP32 full tensor : {fp32_full / mib:.0f} MiB')
print(f'FP32 row tensor  : {fp32_row / kib:.0f} KiB')
print(f'Native cumulative allocations: {native_bytes / mib:.3f} MiB')
print(f'Fused cumulative allocations : {fused_bytes / mib:.3f} MiB')
print(f'Theoretical saving            : {saved_bytes / mib:.3f} MiB')


# 3. 基于 1.6 TB/s 峰值带宽的理想传输时间估算
# 1.6 TB/s 是 Atlas A2 / Ascend 950 官方列出的 HBM 峰值带宽（十进制字节计算）。
# 实际 kernel 受访存模式、bank conflict 等影响，通常达不到该数值。
# 
# one-way：若节省的 64 MiB 仅需从 GM 读出一次（即只考虑输出流量），理想耗时约 42 us。
# write+read：若被消除的中间张量原本需要“上一算子写入 GM → 下一算子读出”，
#             则按两倍流量估算，约 84 us。
gm_bandwidth = 1.6e12
one_way_us = saved_bytes / gm_bandwidth * 1e6
print(f'1.6 TB/s 单向理想传输时间: {one_way_us:.2f} us')
print(f'1.6 TB/s 读写双向理想传输时间: {2 * one_way_us:.2f} us')

In [ ]:
# 用标量 add 模拟 RMSNorm 的 8→1 launch；标量计算量可以忽略不计，主要观察下发开销。
import statistics
import time

import torch
import torch_npu  # noqa: F401，注册 NPU backend

torch.npu.set_device(0)


def measure_chain(launches, *, out_of_place, iterations=5000, repeats=3):
    # 测量每组 launch 链的平均耗时，并取多次测量的中位数。
    times = []
    for _ in range(repeats):
        x = torch.zeros(1, device='npu:0')
        # 预热 NPU，避免首次编译和初始化影响计时。
        for _ in range(100):
            for _ in range(launches):
                x = x + 1 if out_of_place else x.add_(1)
        # 计时前后同步，确保统计完整执行时间。
        torch.npu.synchronize()
        start = time.perf_counter()
        for _ in range(iterations):
            for _ in range(launches):
                x = x + 1 if out_of_place else x.add_(1)
        torch.npu.synchronize()
        times.append((time.perf_counter() - start) * 1e6 / iterations)
    return statistics.median(times)


# in-place：尽量只观察 launch；out-of-place：同时包含输出张量申请。
for label, out_of_place in (
    ('launch-only simulation (in-place)', False),
    ('launch + allocation simulation (out-of-place)', True),
):
    # 原生 RMSNorm 用 8 次 launch，融合路径用 1 次，理论上消除 7 次。
    native_us = measure_chain(8, out_of_place=out_of_place)
    fused_us = measure_chain(1, out_of_place=out_of_place)
    saved_us = native_us - fused_us
    print(f'\n{label}')
    print(f'  8 launches: {native_us:.2f} us/iteration')
    print(f'  1 launch  : {fused_us:.2f} us/iteration')
    print(f'  saved     : {saved_us:.2f} us/iteration')
    print(f'  simulation per eliminated launch: {saved_us / 7:.2f} us')


### 实验结论

结合实验结果，RMSNorm 这类 vector 融合算子的主要收益可归结为：

**纯调度收益**：在 in-place（不申请新张量）场景下，8 次串行 launch 约 115 μs，单次约 15 μs；减少 7 次 launch 可节省约 100 μs，平均每次约 14 μs。这说明减少 launch 次数本身就是显著的优化来源。

**内存分配额外开销**：引入 out-of-place（中间结果需写 GM 再被下一算子读）张量申请后，8-launch 路径涨至约 141 μs，1-launch 路径涨至约 18.5 μs。与 in-place 基线对比：8-launch 路径多出约 26 μs，1-launch 路径仅多出约 3.6 μs。剥离后可得：7 次额外的内存分配/释放共引入约 22.5 μs 的软件层开销（含内存池锁、地址映射、引用计数维护等），分摊每次约 3.2 μs。

作为参照，若原生 RMSNorm 路径中 8 次中间物化对应的 64.008 MiB 数据在硬件层发生完全的 GM 间双向搬运，按 1.6 TB/s 带宽理论极限需约 83.90 μs。然而，这 83.90 μs 并不能直接作为融合算子的预期收益上限。在实际的 PyTorch + CANN 运行时环境中，原生路径的单个算子下发并非“裸跑”在硬件上：PyTorch 和 CANN 运行时确实存在缓存分配器等优化机制，从而在实际执行中规避掉一部分物理层数据搬移。因此，融合算子不能完全获得访存次数减少带来的收益。

综上，融合算子的收益 ≈ 消除多次 launch（~100 μs）+ 消除原生路径中残留的中间张量管理开销（~22.5 μs）。我们将在下一章的 Profiler 实测中给出完整验证。



### 收益来源总结

| | 原生路径 | 融合路径 |
|--|---------|---------|
| Python 调用 | `F.rms_norm(...)` | `torch_npu.npu_rms_norm(...)` |
| 当前 `(2,1024,2048)` 前向 ACLNN 链 | 8 次 launch | 1 次 launch，减少 7 次 |
| launch-only 代理耗时 | 实测约 `114.99 us` | 实测约 `14.88 us`；按 $7\times14.30$ 估算节省约 `100.1 us` |
| 理论累计张量申请 | `72 MiB + 16 KiB` | `8 MiB + 8 KiB`，少约 `64 MiB + 8 KiB` |
| 1.6 TB/s 理想带宽等效 | — | 单向约 `42 us`；写后再读代理约 `84 us` |
| API 输出 | `y` | `y` 与供反向使用的 `rstd` |
| 数学结果 | $y_i=\gamma_i x_i / \sqrt{\operatorname{mean}(x^2)+\varepsilon}$ | 同一公式 |


具体到 Qwen3 训练任务中 RmsNorm算子的真实收益数值，以及融合后的 kernel 实际执行时间，将由下一节的 Profiler 实测数据给出最终验证。

---

## 2. 注册 npu_rms_norm

torchtitan-npu 通过 **ModelConverter** 机制实现融合算子的自动替换。该机制提供一个 `convert` 接口，允许用户自定义模块替换逻辑，在模型构建后、并行化与 optimizer 构建前执行。

### Converter 的工作原理
![](./images/modelconverter-workflow.png)

Converter 遍历 `model.named_modules()`，按类型匹配找到原生 RMSNorm 实例，将其原地替换为 NPURMSNorm，并沿用原有的 `weight` Parameter——这意味着替换不改变参数集合，optimizer 状态不受影响。

关键执行时机：converter 在**并行化（FSDP/TP/PP）和 optimizer 构建之前**运行。这保证了：
- 替换发生在参数被 shard 之前，避免并行化后难以遍历模块
- `weight` Parameter 在替换时直接传递，optimizer 看到的是同一份参数

### 注册与启用

融合算子通过两步启用：

1. **注册**：用 `@register_model_converter("npu_rms_norm")` 将 converter 绑定到一个名字，框架启动时按名字查找
2. **配置**：在训练配置的 `model_converters` 列表中通过 `get_model_converter_config("npu_rms_norm")` 引用该名字，一行即可启用

本教程先保持 `sft_qwen3_1_7b_wordle()` 的 `model_converters` 为空并运行 baseline；随后按下一节说明，将 `_qwen3_1_7b_converters()` 改为引用 `get_model_converter_config("npu_rms_norm")`，重新运行并确认 Qwen3-1.7B 的 113 个目标模块均被替换。


下一节给出 baseline 命令和待 recipe 合入后执行的 fused 命令，并用 profiling 验证效果。

## 练习

1. （单选题）关于 `torch_npu.npu_rms_norm` 的内存访问，哪项表述可由当前接口和 trace 支持？
    A. API 返回 `y` 和 `rstd`；实际 HBM 流量与 kernel 数需按版本实测
    B. HBM 流量固定降到原来的 1/4
    C. 只写回 `y`，`rstd` 不会产生输出
    D. 完全不需要访问 HBM

2. （单选题）`isinstance(module, RMSNorm)` 会匹配哪些模块？
    A. 只匹配精确类型
    B. 匹配 RMSNorm 及其子类
    C. 匹配所有 Module
    D. 只匹配 NPURMSNorm

3. （单选题）复用原 `weight` Parameter 的关键执行时机是什么？
    A. converter 在并行化和 optimizer 构建前执行
    B. optimizer step 之后替换
    C. checkpoint 保存后替换
    D. 推理服务退出后替换

4. （单选题）融合 converter 的生效顺序是哪一项？
    A. import 注册 → 配置构建 → converter 执行 → 并行化/optimizer 构建
    B. optimizer 构建 → import 注册 → 数据下载
    C. profiler 分析 → converter 执行 → 模型构建
    D. 推理启动 → checkpoint 保存 → converter 执行


In [ ]:
%cd $original_dir
!cat ./answer/04.02_answer.txt